In [ ]:
target_dir = "../data/raw/financial-fraud-detection-dataset"

In [5]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils import resample


from tensorflow.keras.regularizers import l2  as L2
from tensorflow.keras.initializers import HeNormal

In [ ]:
csv_file = os.path.join(target_dir, "Synthetic_Financial_datasets_log.csv")
print(f"Looking for file at: {csv_file}")


df = pd.read_csv(csv_file)
print(f'\n')
print(df.head())

Looking for file at: ./Synthetic_Financial_datasets_log.csv


   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  


In [7]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='object')

In [8]:
# remove unuse columns
df.drop(['oldbalanceOrg','newbalanceOrig','oldbalanceDest','newbalanceDest'], axis=1, inplace=True)

In [9]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'nameDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='object')

In [10]:
encoder = OneHotEncoder(sparse_output=False)
columns_to_encode = ['type']

encoded = encoder.fit_transform(df[columns_to_encode])
encoded_columns = encoder.get_feature_names_out(columns_to_encode)
encoded_df = pd.DataFrame(encoded, columns=encoded_columns)

df = pd.concat([df, encoded_df], axis=1)
print(df.head())

   step      type    amount     nameOrig     nameDest  isFraud  \
0     1   PAYMENT   9839.64  C1231006815  M1979787155        0   
1     1   PAYMENT   1864.28  C1666544295  M2044282225        0   
2     1  TRANSFER    181.00  C1305486145   C553264065        1   
3     1  CASH_OUT    181.00   C840083671    C38997010        1   
4     1   PAYMENT  11668.14  C2048537720  M1230701703        0   

   isFlaggedFraud  type_CASH_IN  type_CASH_OUT  type_DEBIT  type_PAYMENT  \
0               0           0.0            0.0         0.0           1.0   
1               0           0.0            0.0         0.0           1.0   
2               0           0.0            0.0         0.0           0.0   
3               0           0.0            1.0         0.0           0.0   
4               0           0.0            0.0         0.0           1.0   

   type_TRANSFER  
0            0.0  
1            0.0  
2            1.0  
3            0.0  
4            0.0  


In [11]:
labels = ['isFraud', 'isFlaggedFraud']
features = ['step', 'amount'] + encoded_columns.tolist()
features
train_size=0.7
train_index_split = int(train_size*df.shape[0])

train, test = df[:train_index_split], df[train_index_split:]

train_min_class_size = train['isFraud'].value_counts().min()

train_rebalanced = (
      train.groupby('isFraud', group_keys=False)
        .apply(lambda x: x.sample(train_min_class_size, random_state=42))
)

print(train_rebalanced['isFraud'].value_counts())

## Balanced test dataset
# test_min_class_size = test['isFraud'].value_counts().min()
# test = (
#       test.groupby('isFraud', group_keys=False)
#         .apply(lambda x: x.sample(test_min_class_size, random_state=42))
# )

print(test['isFraud'].value_counts())

isFraud
0    3643
1    3643
Name: count, dtype: int64
isFraud
0    1904216
1       4570
Name: count, dtype: int64


/tmp/ipython-input-3434896246.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(train_min_class_size, random_state=42))


In [ ]:
def add_all_features(df, train_fraud_series=None):
    """Add all engineered features"""
    df = df.copy()

    # Amount features
    df['amount_log'] = np.log1p(df['amount'])
    df['amount_sqrt'] = np.sqrt(df['amount'])
    df['is_round_amount'] = (df['amount'] % 10000 == 0).astype(int)
    df['is_large_amount'] = (df['amount'] > 200000).astype(int)
    df['amount_zscore'] = (df['amount'] - df['amount'].mean()) / df['amount'].std()

    # Time features
    df['hour_of_day'] = df['step'] % 24
    df['day_of_month'] = (df['step'] // 24) % 30
    df['is_night'] = ((df['hour_of_day'] >= 0) & (df['hour_of_day'] <= 6)).astype(int)

    # Interaction features
    df['amount_x_transfer'] = df['amount'] * df['type_TRANSFER']
    df['amount_x_cashout'] = df['amount'] * df['type_CASH_OUT']

    # FIXED: Convert to int before using bitwise OR, or use addition
    # Option 1: Use addition (if one is 1, result is at least 1)
    df['transfer_or_cashout'] = ((df['type_TRANSFER'] + df['type_CASH_OUT']) > 0).astype(int)

    # Option 2: Or use logical OR with astype
    # df['transfer_or_cashout'] = ((df['type_TRANSFER'].astype(bool)) | (df['type_CASH_OUT'].astype(bool))).astype(int)

    return df

# Apply to datasets
print("\n" + "="*60)
print("STEP 2: ENGINEERING FEATURES")
print("="*60)

train_rebalanced = add_all_features(train_rebalanced)
test_enhanced = add_all_features(test)

print("✓ Sample Features engineered successfully")



# Define all features
all_features = [
    'step', 'amount', 'amount_log', 'amount_sqrt', 'amount_zscore',
    'is_round_amount', 'is_large_amount',
    'hour_of_day', 'day_of_month', 'is_night',
    'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER',
    'amount_x_transfer', 'amount_x_cashout', 'transfer_or_cashout'
]


STEP 2: ENGINEERING FEATURES
✓ Features engineered successfully


In [13]:
def target_encode(X_train_col, y_train, X_test_col, smoothing=10):
    global_mean = y_train.mean()
    temp_df = pd.DataFrame({'col': X_train_col, 'target': y_train})
    agg = temp_df.groupby('col')['target'].agg(['mean', 'count'])
    smoothed_mean = (agg['mean'] * agg['count'] + global_mean * smoothing) / (agg['count'] + smoothing)
    train_encoded = X_train_col.map(smoothed_mean).fillna(global_mean)
    test_encoded = X_test_col.map(smoothed_mean).fillna(global_mean)
    return train_encoded, test_encoded

train_rebalanced['nameOrig_enc'], test_enhanced['nameOrig_enc'] = \
    target_encode(train_rebalanced['nameOrig'], train_rebalanced['isFraud'],
                  test['nameOrig'], smoothing=10)

train_rebalanced['nameDest_enc'], test_enhanced['nameDest_enc'] = \
    target_encode(train_rebalanced['nameDest'], train_rebalanced['isFraud'],
                  test['nameDest'], smoothing=10)

all_features += ['nameOrig_enc', 'nameDest_enc']

print(f"Total features: {len(all_features)}")

# Prepare data
X_train_final = train_rebalanced[all_features].copy()
y_train_final = train_rebalanced['isFraud'].copy()
X_test_final = test_enhanced[all_features].copy()
y_test_final = test_enhanced['isFraud'].copy()

Total features: 20


In [14]:
print("\nScaling features...")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final)
X_test_scaled = scaler.transform(X_test_final)

X_train_scaled = np.nan_to_num(X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
X_test_scaled = np.nan_to_num(X_test_scaled, nan=0.0, posinf=0.0, neginf=0.0)

y_train_final = y_train_final.values.astype(int)
y_test_final = y_test_final.values.astype(int)

# Step 4: Train/val split
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_scaled, y_train_final,
    test_size=0.2,
    random_state=42,
    stratify=y_train_final
)

print(f"Train: {X_train_split.shape}, Fraud rate: {y_train_split.mean():.4f}")
print(f"Val: {X_val_split.shape}, Fraud rate: {y_val_split.mean():.4f}")
print(f"Test: {X_test_scaled.shape}, Fraud rate: {y_test_final.mean():.4f}")


Scaling features...
Train: (5828, 20), Fraud rate: 0.5000
Val: (1458, 20), Fraud rate: 0.5000
Test: (1908786, 20), Fraud rate: 0.0024


In [15]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train_split)
class_weight_values = compute_class_weight('balanced', classes=classes, y=y_train_split)
class_weights = dict(zip(classes.astype(int), class_weight_values))

print(f"\nClass weights: {class_weights}")


Class weights: {np.int64(0): np.float64(1.0), np.int64(1): np.float64(1.0)}


In [17]:
model= Sequential([
    Dense(128, input_shape=(20,), activation='leaky_relu', kernel_regularizer=L2(0.001), kernel_initializer=HeNormal()),
    Dropout(0.1),
    Dense(64, activation='leaky_relu', kernel_regularizer=L2(0.001)),
    Dropout(0.2),
    Dense(32, activation='leaky_relu', kernel_regularizer=L2(0.001)),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall'),
             tf.keras.metrics.AUC(name='auc')]
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         2,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,057 (51.00 KB)

 Trainable params: 13,057 (51.00 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
print("\nTraining model...")

history = model.fit(
    X_train_split, y_train_split,
    validation_data=(X_val_split, y_val_split),
    epochs=100,
    batch_size=256,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)
    ],
    verbose=1
)



Training model...
Epoch 1/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 46ms/step - accuracy: 0.6708 - auc: 0.8337 - loss: 1.0066 - precision: 0.6179 - recall: 0.9743 - val_accuracy: 0.9986 - val_auc: 0.9997 - val_loss: 0.5315 - val_precision: 0.9973 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 2/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9821 - auc: 0.9975 - loss: 0.5393 - precision: 0.9741 - recall: 0.9908 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 0.4199 - val_precision: 1.0000 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 3/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9971 - auc: 0.9995 - loss: 0.4345 - precision: 0.9963 - recall: 0.9980 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 0.3902 - val_precision: 1.0000 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 4/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9996 - auc: 1.0000 - loss: 0.4002 - precision: 0.9997 - recall: 0.9996 - val_accuracy: 1.0000 - val_auc: 1.

In [19]:
print("\nEvaluating model on test set...")

# Predict probabilities
y_pred_proba = model.predict(X_test_scaled).flatten()
y_pred = (y_pred_proba >= 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_test_final, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_final, y_pred))

if len(np.unique(y_test_final)) > 1:
    print(f"\nTest AUC: {roc_auc_score(y_test_final, y_pred_proba):.4f}")


Evaluating model on test set...
59650/59650 ━━━━━━━━━━━━━━━━━━━━ 66s 1ms/step

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.91      0.95   1904216
           1       0.02      0.67      0.04      4570

    accuracy                           0.91   1908786
   macro avg       0.51      0.79      0.50   1908786
weighted avg       1.00      0.91      0.95   1908786


Confusion Matrix:
[[1741435  162781]
 [   1525    3045]]

Test AUC: 0.9091


In [20]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
results = []

for threshold in thresholds:
    y_pred_thresh = (y_pred_proba >= threshold).astype(int)

    precision = precision_score(y_test_final, y_pred_thresh, zero_division=0)
    recall = recall_score(y_test_final, y_pred_thresh, zero_division=0)
    f1 = f1_score(y_test_final, y_pred_thresh, zero_division=0)
    frauds_detected = np.sum(y_pred_thresh == 1)

    results.append({
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'frauds_detected': frauds_detected
    })

    print(f"\nThreshold: {threshold:.1f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  Frauds flagged: {frauds_detected:,}")

# Find optimal threshold (maximize F1)
results_df = pd.DataFrame(results)
best_threshold = results_df.loc[results_df['f1'].idxmax()]
print(f"\n✓ Optimal threshold (max F1): {best_threshold['threshold']:.2f}")
print(f"  F1-Score: {best_threshold['f1']:.4f}")



Threshold: 0.3
  Precision: 0.0089
  Recall:    0.8573
  F1-Score:  0.0177
  Frauds flagged: 438,204

Threshold: 0.4
  Precision: 0.0127
  Recall:    0.7659
  F1-Score:  0.0251
  Frauds flagged: 274,615

Threshold: 0.5
  Precision: 0.0184
  Recall:    0.6663
  F1-Score:  0.0357
  Frauds flagged: 165,826

Threshold: 0.6
  Precision: 0.0245
  Recall:    0.5840
  F1-Score:  0.0469
  Frauds flagged: 109,135

Threshold: 0.7
  Precision: 0.0333
  Recall:    0.4965
  F1-Score:  0.0623
  Frauds flagged: 68,226

Threshold: 0.8
  Precision: 0.0547
  Recall:    0.4007
  F1-Score:  0.0962
  Frauds flagged: 33,500

✓ Optimal threshold (max F1): 0.80
  F1-Score: 0.0962


In [21]:
# ============================================================================
# OPTIONAL: Save Model
# ============================================================================

# Uncomment to save the model
# model.save('fraud_detection_lstm_model.h5')
# print("\nModel saved to 'fraud_detection_lstm_model.h5'")